# Telecom KPI Anomaly Detection Demo

This notebook demonstrates a GitHub-safe anomaly detection workflow for hourly telecom KPI monitoring.

It covers:
- loading sample hourly KPI data
- calculating KPI rates from raw counters
- building rolling historical baselines
- scoring anomalies using rolling median and MAD
- reviewing flagged KPI anomalies by cell and hour


## 1. Imports


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt


## 2. Configuration


In [ ]:
DATA_FILE = Path('data/sample_hourly_kpi.csv')
LOOKBACK_HOURS = 168
MIN_HISTORY_POINTS = 48
EPSILON = 1e-6

KPI_COLUMNS = [
    'rrc_success_rate',
    'e_rab_success_rate',
    'dl_prb_utilization',
    'dl_throughput_mbps',
]

LOWER_IS_BAD_KPIS = {
    'rrc_success_rate',
    'e_rab_success_rate',
    'dl_throughput_mbps',
}

UPPER_IS_BAD_KPIS = {
    'dl_prb_utilization',
}

DEFAULT_THRESHOLD_BY_KPI = {
    'rrc_success_rate': 3.5,
    'e_rab_success_rate': 3.5,
    'dl_prb_utilization': 3.5,
    'dl_throughput_mbps': 3.5,
}


## 3. Helper functions


In [ ]:
def safe_rate(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    denominator = denominator.astype(float)
    numerator = numerator.astype(float)
    return np.where(denominator > 0, (numerator / denominator) * 100.0, np.nan)


def calculate_kpis(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out['timestamp'] = pd.to_datetime(out['timestamp'])
    out['rrc_success_rate'] = safe_rate(out['rrc_success'], out['rrc_attempts'])
    out['e_rab_success_rate'] = safe_rate(out['e_rab_success'], out['e_rab_attempts'])
    out = out.sort_values(['cell_id', 'timestamp']).reset_index(drop=True)
    return out


def add_rolling_baseline(df: pd.DataFrame, value_col: str, lookback_hours: int, min_history_points: int) -> pd.DataFrame:
    work = df.copy()
    median_col = f'{value_col}_baseline_median'
    mad_col = f'{value_col}_baseline_mad'
    mean_col = f'{value_col}_baseline_mean'
    std_col = f'{value_col}_baseline_std'
    count_col = f'{value_col}_history_count'

    def per_cell(group: pd.DataFrame) -> pd.DataFrame:
        s = group[value_col].astype(float)
        shifted = s.shift(1)
        rolling_obj = shifted.rolling(window=lookback_hours, min_periods=min_history_points)

        baseline = pd.DataFrame(index=group.index)
        baseline[median_col] = rolling_obj.median()
        baseline[mad_col] = rolling_obj.apply(lambda x: np.median(np.abs(x - np.median(x))), raw=True)
        baseline[mean_col] = rolling_obj.mean()
        baseline[std_col] = rolling_obj.std(ddof=0)
        baseline[count_col] = rolling_obj.count()

        baseline[mad_col] = baseline[mad_col].fillna(0.0)
        baseline[std_col] = baseline[std_col].fillna(0.0)
        return baseline

    baseline_df = (
        work.groupby('cell_id', group_keys=False, observed=True)
        .apply(per_cell)
        .reset_index(level=0, drop=True)
    )

    work = pd.concat([work, baseline_df], axis=1)
    work[mad_col] = work[mad_col].clip(lower=EPSILON)
    work[std_col] = work[std_col].clip(lower=EPSILON)
    return work


def add_anomaly_score(df: pd.DataFrame, value_col: str, threshold: float) -> pd.DataFrame:
    out = df.copy()
    median_col = f'{value_col}_baseline_median'
    mad_col = f'{value_col}_baseline_mad'
    count_col = f'{value_col}_history_count'
    score_col = f'{value_col}_anomaly_score'
    diff_col = f'{value_col}_baseline_diff'
    flag_col = f'{value_col}_is_anomaly'
    direction_col = f'{value_col}_anomaly_direction'

    out[diff_col] = out[value_col] - out[median_col]
    out[score_col] = np.abs(out[diff_col]) / (1.4826 * out[mad_col].clip(lower=EPSILON))

    def classify_direction(diff: float):
        if pd.isna(diff):
            return None
        if diff > 0:
            return 'up'
        if diff < 0:
            return 'down'
        return 'flat'

    out[direction_col] = out[diff_col].apply(classify_direction)

    if value_col in LOWER_IS_BAD_KPIS:
        bad_side = out[diff_col] < 0
    elif value_col in UPPER_IS_BAD_KPIS:
        bad_side = out[diff_col] > 0
    else:
        bad_side = pd.Series(True, index=out.index)

    enough_history = out[count_col].fillna(0) > 0
    out[flag_col] = (out[score_col] >= threshold) & bad_side & enough_history
    return out


def build_anomaly_summary(df: pd.DataFrame) -> pd.DataFrame:
    records = []
    for kpi in KPI_COLUMNS:
        flag_col = f'{kpi}_is_anomaly'
        score_col = f'{kpi}_anomaly_score'
        diff_col = f'{kpi}_baseline_diff'
        direction_col = f'{kpi}_anomaly_direction'
        median_col = f'{kpi}_baseline_median'

        subset = df.loc[df[flag_col]].copy()
        if subset.empty:
            continue

        subset['kpi_name'] = kpi
        subset['kpi_value'] = subset[kpi]
        subset['baseline_value'] = subset[median_col]
        subset['anomaly_score'] = subset[score_col]
        subset['baseline_diff'] = subset[diff_col]
        subset['anomaly_direction'] = subset[direction_col]

        records.append(subset[[
            'timestamp',
            'cell_id',
            'kpi_name',
            'kpi_value',
            'baseline_value',
            'baseline_diff',
            'anomaly_score',
            'anomaly_direction',
        ]])

    if not records:
        return pd.DataFrame(columns=[
            'timestamp', 'cell_id', 'kpi_name', 'kpi_value',
            'baseline_value', 'baseline_diff', 'anomaly_score', 'anomaly_direction'
        ])

    summary_df = pd.concat(records, ignore_index=True)
    return summary_df.sort_values(['timestamp', 'cell_id', 'kpi_name']).reset_index(drop=True)


## 4. Load raw sample data


In [ ]:
df_raw = pd.read_csv(DATA_FILE)
print(df_raw.shape)
df_raw.head()


## 5. Calculate KPIs


In [ ]:
df_kpi = calculate_kpis(df_raw)
df_kpi[['timestamp', 'cell_id', 'rrc_success_rate', 'e_rab_success_rate', 'dl_prb_utilization', 'dl_throughput_mbps']].head()


## 6. Add rolling baselines and anomaly scores


In [ ]:
df_result = df_kpi.copy()

for kpi in KPI_COLUMNS:
    df_result = add_rolling_baseline(
        df=df_result,
        value_col=kpi,
        lookback_hours=LOOKBACK_HOURS,
        min_history_points=MIN_HISTORY_POINTS,
    )
    df_result = add_anomaly_score(
        df=df_result,
        value_col=kpi,
        threshold=DEFAULT_THRESHOLD_BY_KPI[kpi],
    )

df_result.head()


## 7. Build anomaly summary


In [ ]:
anomaly_summary = build_anomaly_summary(df_result)
print(f'Total anomalies flagged: {len(anomaly_summary)}')
anomaly_summary.head(20)


## 8. KPI-level anomaly counts


In [ ]:
anomaly_summary['kpi_name'].value_counts()


## 9. Review one cell and KPI visually


In [ ]:
cell_to_plot = 'CELL_001'
kpi_to_plot = 'rrc_success_rate'

plot_df = df_result.loc[df_result['cell_id'] == cell_to_plot].copy()
baseline_col = f'{kpi_to_plot}_baseline_median'
flag_col = f'{kpi_to_plot}_is_anomaly'

plt.figure(figsize=(12, 5))
plt.plot(plot_df['timestamp'], plot_df[kpi_to_plot], label=kpi_to_plot)
plt.plot(plot_df['timestamp'], plot_df[baseline_col], label='rolling_baseline')

anomaly_points = plot_df.loc[plot_df[flag_col]]
plt.scatter(anomaly_points['timestamp'], anomaly_points[kpi_to_plot], label='anomalies')

plt.title(f'{cell_to_plot} - {kpi_to_plot}')
plt.xlabel('timestamp')
plt.ylabel(kpi_to_plot)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Export outputs


In [ ]:
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)

df_result.to_csv(output_dir / 'hourly_kpi_anomaly_output.csv', index=False)
anomaly_summary.to_csv(output_dir / 'anomaly_summary.csv', index=False)

print('Files exported to outputs/')
